# Apply a saved model

Labels a match with a model saved in `models/<MODEL_NAME>/`, without retraining, and scores the labels when a ground
truth is available.

* Another match: set `TRACKING` to its raw tracking, a `TrackingData` as returned by `load_atd_tracking`.
* More ground truth: set `GT` to any `frame,label` CSV, with `X` where it has no label. The notebook scores the labels over
  all labelled frames, whether that is 5 minutes or a whole match, and per `CHUNK_S` seconds when the file is longer.
* The bundle holds the CatBoost model, feature columns, temperature, decoder and the feature / preprocessing settings,
  so applying it gives the same labels as the notebook that saved it.

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"     

def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_model.ipynb", globals())
load_notebook_defs("helper_decoder.ipynb", globals())
load_notebook_defs("helper_reporting.ipynb", globals())

In [ ]:
MODEL_NAME = "possession_v1"                         # models/<MODEL_NAME>/ (saved by run_modeling.ipynb)
TRACKING = None                                      # None = BAR-ATH (data/tactical). Another match: a TrackingData, e.g. TrackingData.load(ARTIFACT_DIR / "other_match_raw.npz")
GT = "public"                                        # "public" = the ground truth in evaluation/gt/ (BAR-ATH only) | a path to any frame,label CSV | None = do not score
CHUNK_S = 600.0                                      # a longer ground truth is also scored per chunk of this many seconds (0 = no chunks)
SUBMISSION_PATH = REPO_ROOT / "output" / "FORZA INDO.csv"      # where the labels are written (a frame,label CSV for the whole tracking)

In [ ]:
bundle = load_bundle(MODEL_NAME)
print("saved:", bundle["meta"]["saved"], "| trees:", bundle["meta"]["trees"], "| temperature:", round(bundle["meta"]["temperature"], 3), "| trained with:", bundle["meta"]["info"])
raw = TRACKING if TRACKING is not None else load_atd_tracking(get_paths())
if TRACKING is None:
    clean = load_or_build_clean(raw, bundle["pre_cfg"])                                                     # cached for BAR-ATH
else:
    clean = TrackingPreprocessor(dataclasses.replace(bundle["pre_cfg"], pitch_m=tuple(raw.pitch_m))).run(raw, verbose=False)   # any other match: never overwrites the BAR-ATH cache
T = clean.n_frames
print(f"frames={T} | graded={int((clean.half_id() > 0).sum())}")

In [ ]:
pred = predict_bundle(bundle, clean)
labels, probs = pred["labels"], pred["probs"]
print("predicted")

In [ ]:
if GT is not None:
    gt = load_public_gt(get_paths()["gt_public"] if GT == "public" else Path(GT), T)
    _ = evaluate_on_gt(labels, probs, gt, clean, tag="predict", name=MODEL_NAME, chunk_s=CHUNK_S)
else:
    print("no ground truth given: the labels are written, nothing is scored")